In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="hon9kon9ize/yue_emo_speech", 
    repo_type="dataset", local_dir="./yue_emo_speech", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 45 files: 100%|██████████| 45/45 [00:00<00:00, 3541.49it/s]


'/home/ubuntu/yue_emo_speech'

In [3]:
files = glob('yue_emo_speech/*/*.parquet')
len(files)

45

In [4]:
df = pd.read_parquet(files[0])
df

,audio_file,duration,label,confidence,text
0,{'bytes': b'RIFF>\xb3\x04\x00WAVEfmt \x10\x00\...,111744,neutral,0.999872,我哋甚至乎喺心入面可以同时接纳两种截然相反嘅观点，两种截然相反嘅立场。
1,{'bytes': b'RIFF\xfeD\x04\x00WAVEfmt \x10\x00\...,101504,neutral,0.803620,唔咪住先，我想翻翻到关于喺唔同层面度做决定嗰句说话，你头先讲过。
2,{'bytes': b'RIFFH9\x04\x00WAVEfmt \x10\x00\x00...,100417,neutral,0.998114,我哋每日都要同司姐 讲，我喺一段时间入面只可以专注做某一件我最擅长嘅事情。
3,{'bytes': b'RIFF\xf4\x8a\x03\x00WAVEfmt \x10\x...,84225,neutral,0.998983,咁，呢间公司就会越嚟越强第七个思考咧，就系要将想做嘅事情当成个工作。😊
4,{'bytes': b'RIFFbZ\x02\x00WAVEfmt \x10\x00\x00...,55937,happy,0.956796,但系有一啲思想搜救嘅大臣。😡
...,...,...,...,...,...
16141,{'bytes': b'RIFF\xce\x15\x03\x00WAVEfmt \x10\x...,73344,neutral,0.992710,你嘅内在小孩安定啦，佢觉得温暖啦，佢觉得。
16142,{'bytes': b'RIFFXm\x01\x00WAVEfmt \x10\x00\x00...,33920,happy,1.000000,佢就跳咗入海里面。😮
16143,{'bytes': b'RIFF\x84\xba\x01\x00WAVEfmt \x10\x...,41088,happy,0.995229,对外在知识同埋规律嘅追求。
16144,{'bytes': b'RIFF\xa8O\x05\x00WAVEfmt \x10\x00\...,126272,neutral,0.996797,后世嘅人虽然强行咁加入咗一啲宗教嘅色彩入去，但系其实，佢嘅本质咧依然系素论依然系无神论。😊


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio_file'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [10]:
data = multiprocessing(files, loop, cores = 20)

In [7]:
len(data)

726563

In [8]:
with open('yue_emo_speech.json', 'w') as fopen:
    json.dump(data, fopen)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('yue_emo_speech-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
# !zip -rq yue_emo_speech_audio.zip yue_emo_speech_audio

In [15]:
# !hf upload malaysia-ai/Multilingual-TTS yue_emo_speech_audio.zip --repo-type=dataset

In [18]:
# !zip -rq yue_emo_speech_audio_neucodec.zip yue_emo_speech_audio_neucodec

In [19]:
# !hf upload malaysia-ai/Multilingual-TTS yue_emo_speech_audio_neucodec.zip --repo-type=dataset

In [20]:
import json

with open('yue_emo_speech.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 726563/726563 [00:00<00:00, 2594698.00it/s]


726563

In [21]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'yue_emo_speech_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 726563/726563 [03:08<00:00, 3863.17it/s]


In [22]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [23]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'yue_emo_speech_audio/yue_emo_speech-data-train-00038-of-00045_0.mp3',
 'text': '我哋甚至乎喺心入面可以同时接纳两种截然相反嘅观点，两种截然相反嘅立场。',
 'speaker': 'yue_emo_speech_audio_0'}

In [25]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'yue_emo_speech')

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  7.83ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  58%|█████▊    | 25.8MB / 44.3MB,  129MB/s  
Processing Files (0 / 1):  99%|█████████▉| 43.8MB / 44.3MB,  110MB/s  
Processing Files (1 / 1): 100%|██████████| 44.3MB / 44.3MB, 55.5MB/s  
Processing Files (1 / 1): 100%|██████████| 44.3MB / 44.3MB, 44.4MB/s  
New Data Upload: 100%|██████████| 44.3MB / 44.3MB, 44.4MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.97s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/4cc0deca1ee9808daab35433ff29aab2b49cb413', commit_message='Upload dataset', commit_description='', oid='4cc0deca1ee9808daab35433ff29aab2b49cb413', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)